# Part 2: Matching sales to EPCs by address

Joins Price Paid Data to EPCs on address. Tries exact matching on address word sets first, then a subset test, which raised the usable match rate from 63.7% to 67.2%. Also looks at why sales fail to match, and at ambiguous matches.

Findings and decisions: `logs/part_2_regex_parsing.md`

**Note on outputs:** the outputs of this notebook have been cleared because
address matching displays individual address-level records (addresses,
postcodes). The code and commentary are unchanged, and the results
(match rates, funnel figures and failure cases) are summarised in the
markdown cells below and in [`logs/`](../logs/). To see the outputs, run
the notebook locally with the source data (see the README).

#### Load Price data

In [ ]:
import pandas as pd
from matplotlib import pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1000)
pd.set_option('display.expand_frame_repr', False)


folder = Path('../data/raw')
file_gen = folder.glob('ppd_data*.csv')
pp_raw = pd.concat([pd.read_csv(f) for f in file_gen], ignore_index=True)
pp_raw['deed_date'] = pp_raw['deed_date'].astype('datetime64[ns]')
pp_raw.info()

#### Load EPC data

In [ ]:
epc_raw = pd.read_csv('../data/raw/EDC_OXFORDSHIRE_2015_2025.csv', usecols= [
    'certificate_number', 'address1', 'address2', 'address3', 'postcode',
    'posttown', 'address', 'local_authority', 'local_authority_label',
    'built_form', 'construction_age_band', 'current_energy_efficiency',
    'current_energy_rating', 'energy_consumption_current', 'extension_count', 'floor_height', 'heating_cost_current',
    'inspection_date', 'lodgement_date', 'number_habitable_rooms',
    'number_heated_rooms', 'potential_energy_efficiency', 'property_type',
    'tenure', 'total_floor_area', 'transaction_type', 'uprn', 'uprn_source'
])
epc_raw['inspection_date'] = epc_raw['inspection_date'].astype('datetime64[ns]')
epc_raw['lodgement_date'] = epc_raw['lodgement_date'].astype('datetime64[ns]')
epc_raw.info()

In [ ]:
print(pp_raw[['postcode','saon', 'paon', 'street', 'locality', 'town']].head())
print(epc_raw[['postcode', 'address3', 'address2', 'address1', 'posttown']].head())

#### Comparing address data

In [ ]:
PLACEHOLDER_POSTCODE = 'OX1 1AA' # replace with real postcode to compare datasets
print(pp_raw[pp_raw['postcode'] == 'PLACEHOLDER_POSTCODE'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].head())
print(epc_raw[epc_raw['postcode'] == 'PLACEHOLDER_POSTCODE'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].head())

In [ ]:
PLACEHOLDER_POSTCODE = 'OX1 1AA' # replace with real postcode to compare datasets
print(pp_raw[pp_raw['postcode'] == PLACEHOLDER_POSTCODE][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('saon', ascending=True).head(10))
print(epc_raw[epc_raw['postcode'] == PLACEHOLDER_POSTCODE][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=True).head(10))


- Comparison of address data
    - PP full caps vs EPC capital case
    - pp_'locality' is typically epc_'address3' but can be epc_'address2', depending on how the address is split
    - pp_'saon' or pp_'saon'+pp_'paon' can be epc_'address1'
    - pp_'town' == epc_'post_town'
    - Both appear use 'FLAT 8' vs 'Flat 8'

In [ ]:
print(pp_raw[pp_raw['postcode'] == 'MK18 4AS'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('saon', ascending=False).head(10))
print(epc_raw[epc_raw['postcode'] == 'MK18 4AS'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=False).head(10))

- Matches well on postcode and paon = address1, saon is null
- One record where posttown uses the value put in address3 for the other records

In [ ]:
print(pp_raw[pp_raw['postcode'] == 'OX15 0PT'][['postcode', 'locality', 'street', 'saon', 'paon', 'town']].sort_values('paon', ascending=False).head(10))
print(epc_raw[epc_raw['postcode'] == 'OX15 0PT'][['postcode', 'address3', 'address2', 'address1', 'posttown', 'address']].sort_values('address1', ascending=False).head(10))

#### Cleaning

Fix trailing whitespace

In [ ]:
pp = pp_raw.copy()

for c in pp.select_dtypes(include=['string']).columns:
    pp[c] = pp[c].str.strip()
    print(f"{c} stripped")

In [ ]:
epc = epc_raw.copy()

for c in epc.select_dtypes(include=['string']).columns:
    epc[c] = epc[c].str.strip()
    print(f"{c} stripped")

Remove commas

In [ ]:
epc['address1'] = epc['address1'].str.replace(',', '', regex=False)

- For loops over columns are fast (over rows are slow). List comprehensions wont work as they lose the index.

#### Initial join (exact token matches)

Generate combined addresses (space separated) and match cases

In [ ]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality'], pp['postcode']], sep=' ', na_rep='')
print(pp['address'])
epc['address'] = epc['address1'].str.cat([epc['address2'], epc['address3'], epc['postcode']], sep=' ', na_rep='').apply(str.upper)
print(epc['address'])

- The space separator is being added before/after null values (even though they are empty strings)
- Some commas are making it into the combined addresses

In [ ]:
pp['address_tokens'] = pp['address'].str.split().apply(frozenset)
epc['address_tokens'] = epc['address'].str.split().apply(frozenset)
print(epc['address_tokens'])

- Additional spaces are removed during split
- Use .apply(set) rather than set() so that the function is applied to each row of the new series, rather than trying to apply it to the entire series at once

In [ ]:
first_join = pp.merge(epc, on='address_tokens', how='inner')

print(f"""New table has {first_join.shape[0]} rows, but this includes multiple EPC records per property matched. 
      Matched {first_join['unique_id'].nunique()} Price paid records with an EPC certificate. Match rate: {100* first_join['unique_id'].nunique()/pp.shape[0]:.1f}%""")


In [ ]:
print(first_join.sort_index(axis=1).sample(20))

#### Second join

In [ ]:
epc_address_cols = ['address1', 'address2', 'address3', 'postcode']
pp_address_cols = ['paon', 'saon', 'street', 'locality', 'postcode']

print(f"Commas replaced in epc:\n{epc[epc_address_cols].apply(lambda col: col.str.count(',')).sum()}")
print(f"Commas replaced in pp:\n{pp[pp_address_cols].apply(lambda col: col.str.count(',')).sum()}")

for c in epc_address_cols:
    epc[c] = epc[c].str.replace(',', '', regex=False)
for c in pp_address_cols:
    pp[c] = pp[c].str.replace(',', '', regex=False)

In [ ]:
pp['address'] = pp['paon'].str.cat([pp['saon'], pp['street'], pp['locality'], pp['postcode']], sep=' ', na_rep='')
epc['address'] = epc['address1'].str.cat([epc['address2'], epc['address3'], epc['postcode']], sep=' ', na_rep='').apply(str.upper)

pp['address_tokens'] = pp['address'].str.split().apply(frozenset)
epc['address_tokens'] = epc['address'].str.split().apply(frozenset)

second_join = pp.merge(epc, on='address_tokens', how='inner')

print(f"""New table has {second_join.shape[0]} rows, but this includes multiple EPC records per property matched. 
      Matched {second_join['unique_id'].nunique()} unique Price paid records with an EPC certificate. Match rate: {100* second_join['unique_id'].nunique()/pp.shape[0]:.1f}%""")


Filtering EPC matches

In [ ]:
second_join['time_to_sale'] = second_join['deed_date'] - second_join['inspection_date']

pre_epc = second_join[second_join['time_to_sale'] >= '0 days']
print(f"Properties matched with EPCs only before sale: {pre_epc.shape[0]}")

late_epc = second_join[second_join['time_to_sale'] < '0 days']
print(f"Properties matched with EPCs after sale: {late_epc.shape[0]}")

In [ ]:
dup_rows = second_join[['unique_id', 'certificate_number']].groupby('unique_id')['certificate_number'].agg('count')
print(f"Properties with more than 1 EPC: {(dup_rows > 1).sum()}. Rows to remove: {second_join.shape[0] - second_join['unique_id'].nunique()}")

Filtered to properties with EPC after sale and drop all but closest in date

In [ ]:
filtered_out = second_join[second_join['time_to_sale'] < '0 days'].sort_values(by='time_to_sale', ascending=False).drop_duplicates(subset='unique_id')
print(filtered_out.shape[0])

print(filtered_out['time_to_sale'].describe())

- median days after sale is 1418 day (ca. 4 years), 75th percentile of 630 days.
- *Decision: Drop all properties with EPCs after sale. Return to filter more specifically if time allows.*

In [ ]:
drop_neg = second_join[second_join['time_to_sale'] >= '0 days']
second_join_presale_epc = drop_neg.sort_values(by='time_to_sale', ascending=True).drop_duplicates(subset='unique_id')
print(f"Properties remaining with matched pre-sale EPCs: {second_join_presale_epc.shape[0]}")
print(f"Properties matched but with no usable pre-sale EPC: {second_join['unique_id'].nunique() - second_join_presale_epc.shape[0]}")

#### Sampling unmatched records

In [ ]:
unmatched_pp = pp.merge(epc, on='address_tokens', how='left_anti')

unmatched_epc = pp.merge(epc, on='address_tokens', how='right_anti')

epc_address_cols = ['address_y', 'address1', 'address2', 'address3', 'postcode_y']
pp_address_cols = ['address_x', 'paon', 'saon', 'street', 'locality', 'postcode_x', 'deed_date']

print(unmatched_pp[pp_address_cols].sample(20))
print(unmatched_epc[epc_address_cols].sample(20))

In [ ]:
test_pp = unmatched_pp[pp_address_cols]
test_epc = unmatched_epc[epc_address_cols]

postcode_check = test_pp.merge(test_epc, left_on='postcode_x', right_on='postcode_y', how='inner') 

In [ ]:
print(postcode_check.shape[0])

In [ ]:
print('Null in price paid set:')
print(pp.isna().mean())
print()
print('Null in unmatched price paid set:')
print(test_pp.isna().mean())


- non-null saon is enriched in unmatched set

In [ ]:
unmatched_postcodes = pp.merge(epc, on='postcode', how='left_anti')
print(unmatched_postcodes['unique_id'].nunique())

In [ ]:
print(pp['deed_date'].describe())
print(unmatched_pp['deed_date'].describe())

- deed_date distribution is similar in full PP set and unmatched

In [ ]:
print(pd.concat([pp['new_build'].value_counts(normalize=True), unmatched_pp['new_build'].value_counts(normalize=True)], axis=1, keys=['full_pp', 'unmatched_pp']))

- Unmatched are slightly less likely to be new builds

In [ ]:
print(pd.concat([pp['estate_type'].value_counts(normalize=True), unmatched_pp['estate_type'].value_counts(normalize=True)], axis=1, keys=['full_pp', 'unmatched_pp']))

In [ ]:
print(pp['price_paid'].describe())
print(unmatched_pp['price_paid'].describe())

- Unmatched set contains slightly more expensive properties

- Debugging cell deleted. Found record in price paid dataset using postcode and paon and compared to EPC dataset filtered using postcode and address1.
- Mismatch appears to be due to inclusion of town in address3 in the EPC (and not in locality in PP)

- Debugging cell deleted. Found record in price paid dataset using postcode and paon and compared to EPC dataset filtered using postcode and address1.
- Did not match due to inclusion of town in address3 in the EPC (in addition to the additional locality in PP)

#### Third Join (subset matching)

In [ ]:
postcode_match = pp.merge(epc, on='postcode', how='inner')

In [ ]:

def compare(row):
    return row['address_tokens_x'].issubset(row['address_tokens_y'])

postcode_match['result'] = postcode_match.apply(compare, axis=1)

print(postcode_match.head())

In [ ]:
postcode_match['result'].value_counts()

In [ ]:
subset_matches = postcode_match[postcode_match['result']]

print(f"Third join (subset method) gives: {subset_matches['unique_id'].nunique()} matches. A matching rate of {100 * subset_matches['unique_id'].nunique()/pp.shape[0]:.1f}%")

In [ ]:
multi_match = subset_matches.groupby('unique_id').size()

print(multi_match[multi_match > 1])

In [ ]:
multi_match = subset_matches.groupby('unique_id')['uprn'].nunique()

print(multi_match[multi_match > 1])

- Debugging cell deleted. Found record in price paid dataset using uprn and EPC dataset using postcode and compared.
- 'Churchill' subset matched to both correct and APARTMENT with same other tokens

In [ ]:

epc['uprn'] = epc['uprn'].fillna('1').astype('int64')


- Debugging cell deleted. Found record in price paid dataset using uprn and EPC dataset using postcode and compared.
- One of the numbers is lost when a token set is made from address with flat and street number. Which then matches with the other 44 properties at that street number.

In [ ]:
pp['token_count'] = pp['address'].str.count(r"\S+")

print((pp['token_count'] > pp['address_tokens'].apply(len)).sum())

In [ ]:
lost_tokens = pp[pp['token_count'] > pp['address_tokens'].apply(len)]
multi_uprn = multi_match[multi_match > 1]
result = lost_tokens[lost_tokens['unique_id'].isin(multi_uprn.index)]
print(f"Of the 975 unique_ids matched to >1 uprn, {result.shape[0]} or {100 * result.shape[0]/ multi_uprn.shape[0]:.1f}% lost duplicate tokens in token set formation.")

- Needs date filtering

In [ ]:
third_join = subset_matches[~subset_matches['unique_id'].isin(multi_uprn.index)]
print(third_join.shape)

In [ ]:

third_join['time_to_sale'] = third_join['deed_date'] - third_join['inspection_date']

drop_neg = third_join[third_join['time_to_sale'] >= '0 days']
third_join_presale_epc = drop_neg.sort_values(by='time_to_sale', ascending=True).drop_duplicates(subset='unique_id')
print(f"Properties remaining with matched pre-sale EPCs: {third_join_presale_epc.shape[0]}")
print(f"Corresponss to a {100 * third_join_presale_epc.shape[0]/pp.shape[0]:.1f}% match rate")
